# Archived evaluation experiments

Chronological development scratchpad retained for provenance. Cells represent separate experiments and are not intended to run sequentially as one pipeline. Prefer the canonical notebooks under `table2text_pydanticai/evaluation/notebooks/` for reproducible evaluation.


## Initial evaluation framework


In [ ]:
from pathlib import Path
from table2text.evaluation import (
    init_notebook_evaluation,
    prepare_examples_for_notebook,
    generate_reports_for_notebook,
    score_reference_metrics_for_notebook,
    diagnostics_for_notebook,
    aggregate_for_notebook,
)

project_dir = Path("/Users/realgobs/Documents/MScproject/table2text_pydanticai")

In [ ]:
paths = init_notebook_evaluation(project_dir)
paths

In [ ]:
prep = prepare_examples_for_notebook(
    project_dir,
    skip_unavailable=True,
)
prep

In [ ]:
metrics = score_reference_metrics_for_notebook(project_dir)
metrics[["dataset_id", "variant_id", "metric_name", "status", "score"]].head()

In [ ]:
diagnostics = diagnostics_for_notebook(project_dir)
analysis_paths = aggregate_for_notebook(project_dir)

diagnostics.head(), analysis_paths

## Environment installation


In [ ]:
%pip install -e "/Users/realgobs/Documents/MScproject/table2text_pydanticai[dev,evaluation]"

## Smoke-generation experiments


In [ ]:
import json
from pathlib import Path

from table2text.evaluation import (
    default_paths,
    generate_reports_for_notebook,
    score_reference_metrics_for_notebook,
    diagnostics_for_notebook,
)
from table2text.evaluation.datasets import read_examples, write_jsonl

project_dir = Path("/Users/realgobs/Documents/MScproject/table2text_pydanticai")
paths = default_paths(project_dir)

examples = read_examples(paths["prepared_examples"])

one_each = {}
for example in examples:
    one_each.setdefault(example.dataset_id, example)

five_examples = list(one_each.values())[1:2]

smoke_examples_path = project_dir / "evaluation/prepared/smoke_five.jsonl"
write_jsonl(smoke_examples_path, five_examples)

print("Smoke examples:", len(five_examples))
print([example.dataset_id for example in five_examples])

In [ ]:
variants_payload = json.loads(paths["variant_config"].read_text(encoding="utf-8"))

smoke_variants = {
    "variants": [
        {**variant, "enabled": variant["variant_id"] == "full_system"}
        for variant in variants_payload["variants"]
    ]
}

smoke_variants_path = project_dir / "evaluation/config/archive/variants_smoke.json"
smoke_variants_path.write_text(json.dumps(smoke_variants, indent=2), encoding="utf-8")

In [ ]:
smoke_generations_path = project_dir / "evaluation/generations/smoke_five_generations.jsonl"
smoke_run_root = project_dir / "evaluation/generations/smoke_five_runs"

generations = await generate_reports_for_notebook(
    project_dir,
    examples_path=smoke_examples_path,
    variants_path=smoke_variants_path,
    output_path=smoke_generations_path,
    run_root=smoke_run_root,
    resume=False,
)

generations[
    ["dataset_id", "variant_id", "error", "release_status", "writer_mode", "elapsed_seconds"]
]

In [ ]:
scores = score_reference_metrics_for_notebook(
    project_dir,
    generations_path=smoke_generations_path,
    output_path=project_dir / "evaluation/results/smoke_five_reference_metrics_include_ineligible.jsonl",
    include_ineligible=True,
)

scores[["dataset_id", "variant_id", "metric_name", "status", "score"]]

In [ ]:
import json
from pathlib import Path

from table2text.evaluation import (
    default_paths,
    generate_reports_for_notebook,
)
from table2text.evaluation.datasets import read_examples, write_jsonl

project_dir = Path("/Users/realgobs/Documents/MScproject/table2text_pydanticai")
paths = default_paths(project_dir)

examples = read_examples(paths["prepared_examples"])
totto_examples = [example for example in examples if example.dataset_id == "totto"][:1]

smoke_examples_path = project_dir / "evaluation/prepared/totto_smoke_1.jsonl"
write_jsonl(smoke_examples_path, totto_examples)

variants_payload = json.loads(paths["variant_config"].read_text(encoding="utf-8"))
smoke_variants = {
    "variants": [
        {**variant, "enabled": variant["variant_id"] == "full_system"}
        for variant in variants_payload["variants"]
    ]
}

smoke_variants_path = project_dir / "evaluation/config/archive/variants_totto_smoke.json"
smoke_variants_path.write_text(json.dumps(smoke_variants, indent=2), encoding="utf-8")

smoke_generations_path = project_dir / "evaluation/generations/totto_smoke_generations.jsonl"
smoke_run_root = project_dir / "evaluation/generations/totto_smoke_runs"

generations = await generate_reports_for_notebook(
    project_dir,
    examples_path=smoke_examples_path,
    variants_path=smoke_variants_path,
    output_path=smoke_generations_path,
    run_root=smoke_run_root,
    resume=False,
)

generations[
    ["dataset_id", "variant_id", "error", "release_status", "writer_mode", "elapsed_seconds"]
]

In [ ]:
import json
from pathlib import Path

from table2text.evaluation import default_paths, generate_reports_for_notebook
from table2text.evaluation.datasets import read_examples, write_jsonl

project_dir = Path("/Users/realgobs/Documents/MScproject/table2text_pydanticai")
paths = default_paths(project_dir)

examples = read_examples(paths["prepared_examples"])
totto_examples = [example for example in examples if example.dataset_id == "totto"]

# Change this index to test a different ToTTo example.
totto_index = 10
chosen = [totto_examples[totto_index]]

examples_path = project_dir / "evaluation/prepared/totto_one_test.jsonl"
write_jsonl(examples_path, chosen)

variants_payload = json.loads(paths["variant_config"].read_text(encoding="utf-8"))
variants = {
    "variants": [
        {**variant, "enabled": variant["variant_id"] == "full_system"}
        for variant in variants_payload["variants"]
    ]
}

variants_path = project_dir / "evaluation/config/archive/variants_totto_one.json"
variants_path.write_text(json.dumps(variants, indent=2), encoding="utf-8")

generations = await generate_reports_for_notebook(
    project_dir,
    examples_path=examples_path,
    variants_path=variants_path,
    output_path=project_dir / "evaluation/generations/totto_one_test_generations.jsonl",
    run_root=project_dir / "evaluation/generations/totto_one_test_runs",
    resume=False,
)

display(generations[
    ["dataset_id", "example_id", "variant_id", "release_status", "writer_mode", "error"]
])

print("\nGenerated text:\n")
print(generations.iloc[0]["generated_text"])

print("\nRun artifact:\n")
print(generations.iloc[0]["pipeline_result_path"])

In [ ]:
import json
from pathlib import Path

from table2text.evaluation import default_paths, generate_reports_for_notebook
from table2text.evaluation.datasets import read_examples, write_jsonl

project_dir = Path("/Users/realgobs/Documents/MScproject/table2text_pydanticai")
paths = default_paths(project_dir)

examples = read_examples(paths["prepared_examples"])

dataset_id = "e2e_nlg"

dataset_examples = [
    example for example in examples
    if example.dataset_id == dataset_id
]

print("Available examples:", len(dataset_examples))

example_index = 5  # change this: 0, 1, 2, 3, ...
one_example = dataset_examples[example_index]

print("Selected:", one_example.example_id)
print("Source:", one_example.source_text)
print("References:", one_example.references)

smoke_examples_path = project_dir / f"evaluation/prepared/{dataset_id}_example_{example_index}.jsonl"
write_jsonl(smoke_examples_path, [one_example])

variants_payload = json.loads(paths["variant_config"].read_text(encoding="utf-8"))
smoke_variants = {
    "variants": [
        {**variant, "enabled": variant["variant_id"] == "full_system"}
        for variant in variants_payload["variants"]
    ]
}

smoke_variants_path = project_dir / f"evaluation/config/variants_{dataset_id}_one.json"
smoke_variants_path.write_text(json.dumps(smoke_variants, indent=2), encoding="utf-8")

generations_path = project_dir / f"evaluation/generations/{dataset_id}_example_{example_index}_generations.jsonl"
run_root = project_dir / f"evaluation/generations/{dataset_id}_example_{example_index}_runs"

generations = await generate_reports_for_notebook(
    project_dir,
    examples_path=smoke_examples_path,
    variants_path=smoke_variants_path,
    output_path=generations_path,
    run_root=run_root,
    resume=False,
)

generations[
    [
        "dataset_id",
        "example_id",
        "variant_id",
        "generated_text",
        "error",
        "release_status",
        "writer_mode",
    ]
]
print("\nGenerated text:\n")
print(generations.iloc[0]["generated_text"])

print("\nRun artifact:\n")
print(generations.iloc[0]["pipeline_result_path"])

In [ ]:
import json
from pathlib import Path

from table2text.evaluation import default_paths, generate_reports_for_notebook
from table2text.evaluation.datasets import read_examples, write_jsonl

project_dir = Path("/Users/realgobs/Documents/MScproject/table2text_pydanticai")
paths = default_paths(project_dir)

dataset_id = "mlb_data_to_text"
example_index = 10  # change to 1, 2, 3... for a different MLB example

examples_path = project_dir / f"evaluation/prepared/{dataset_id}.jsonl"
examples = read_examples(examples_path)

one_example = examples[example_index]

print("Available examples:", len(examples))
print("Selected:", one_example.example_id)
print("References:", one_example.references[:2])

smoke_examples_path = project_dir / f"evaluation/prepared/{dataset_id}_example_{example_index}.jsonl"
write_jsonl(smoke_examples_path, [one_example])

variants_payload = json.loads(paths["variant_config"].read_text(encoding="utf-8"))
smoke_variants = {
    "variants": [
        {**variant, "enabled": variant["variant_id"] == "full_system"}
        for variant in variants_payload["variants"]
    ]
}

smoke_variants_path = project_dir / f"evaluation/config/variants_{dataset_id}_one.json"
smoke_variants_path.write_text(json.dumps(smoke_variants, indent=2), encoding="utf-8")

generations_path = project_dir / f"evaluation/generations/{dataset_id}_example_{example_index}_generations.jsonl"
run_root = project_dir / f"evaluation/generations/{dataset_id}_example_{example_index}_runs"

generations = await generate_reports_for_notebook(
    project_dir,
    examples_path=smoke_examples_path,
    variants_path=smoke_variants_path,
    output_path=generations_path,
    run_root=run_root,
    resume=False,
)

generations[
    [
        "dataset_id",
        "example_id",
        "variant_id",
        "generated_text",
        "error",
        "release_status",
        "writer_mode",
    ]
]

In [ ]:
import json
from pathlib import Path

from table2text.evaluation import default_paths, generate_reports_for_notebook
from table2text.evaluation.datasets import read_examples, write_jsonl

project_dir = Path("/Users/realgobs/Documents/MScproject/table2text_pydanticai")
paths = default_paths(project_dir)

examples = read_examples(paths["prepared_examples"])

dataset_id = "sportsett_basketball"
example_id = "4934"

one_example = next(
    e for e in examples
    if e.dataset_id == dataset_id and e.example_id == example_id
)

examples_path = project_dir / f"evaluation/prepared/{dataset_id}_{example_id}.jsonl"
write_jsonl(examples_path, [one_example])

variants_payload = json.loads(paths["variant_config"].read_text(encoding="utf-8"))
variants = {
    "variants": [
        {**variant, "enabled": variant["variant_id"] == "full_system"}
        for variant in variants_payload["variants"]
    ]
}

variants_path = project_dir / f"evaluation/config/variants_{dataset_id}_{example_id}.json"
variants_path.write_text(json.dumps(variants, indent=2), encoding="utf-8")

generations_path = project_dir / f"evaluation/generations/{dataset_id}_{example_id}_generations.jsonl"
run_root = project_dir / f"evaluation/generations/{dataset_id}_{example_id}_runs"

generations = await generate_reports_for_notebook(
    project_dir,
    examples_path=examples_path,
    variants_path=variants_path,
    output_path=generations_path,
    run_root=run_root,
    resume=False,
)

generations[
    [
        "dataset_id",
        "example_id",
        "variant_id",
        "generated_text",
        "error",
        "release_status",
        "writer_mode",
        "elapsed_seconds",
    ]
]

In [ ]:
print("\nGenerated text:\n")
print(generations.iloc[0]["generated_text"])

print("\nRun artifact:\n")
print(generations.iloc[0]["pipeline_result_path"])

## Reference and source-grounded metrics


In [ ]:
import json
from pathlib import Path

from table2text.evaluation import (
    default_paths,
    generate_reports_for_notebook,
    score_reference_metrics_for_notebook,
    score_deepeval_for_notebook,
)
from table2text.evaluation.datasets import read_examples, write_jsonl
from table2text.evaluation.generation import read_generations

project_dir = Path("/Users/realgobs/Documents/MScproject/table2text_pydanticai")
paths = default_paths(project_dir)

dataset_id = "sportsett_basketball"
example_id = "4934"

# 1. Select one SportSett example
examples = read_examples(paths["prepared_examples"])
example = next(
    e for e in examples
    if e.dataset_id == dataset_id and e.example_id == example_id
)

examples_path = project_dir / "evaluation/prepared/sportsett_raw_baseline_one.jsonl"
write_jsonl(examples_path, [example])

# 2. Generate raw DeepSeek V4-Pro baseline
variants_path = project_dir / "evaluation/config/variants_raw_deepseek_v4_flash.json"
variants_path.write_text(
    json.dumps(
        {
            "variants": [
                {
                    "variant_id": "raw_deepseek_v4_flash",
                    "enabled": True,
                    "backend": "callable",
                    "description": "Raw single-LLM DeepSeek baseline from source data only.",
                    "settings_overrides": {
                        "raw_baseline_model": "deepseek-v4-flash",
                        "raw_baseline_max_source_characters": 100000,
                        "raw_baseline_max_output_tokens": 3000,
                        "raw_baseline_temperature": 0.2,
                    },
                    "callable_path": "table2text.evaluation_backends.single_agent_baseline",
                    "command": [],
                    "precomputed_path": None,
                    "repetitions": 1,
                    "seeds": [42],
                }
            ]
        },
        indent=2,
    ),
    encoding="utf-8",
)

raw_generations_path = project_dir / "evaluation/generations/sportsett_raw_deepseek_v4_flash_generations.jsonl"
raw_run_root = project_dir / "evaluation/generations/sportsett_raw_deepseek_v4_flash_runs"

raw_generations = await generate_reports_for_notebook(
    project_dir,
    examples_path=examples_path,
    variants_path=variants_path,
    output_path=raw_generations_path,
    run_root=raw_run_root,
    resume=False,
)

display(raw_generations[
    ["dataset_id", "example_id", "variant_id", "generated_text", "error", "elapsed_seconds"]
])


In [ ]:
import json
import random
from pathlib import Path

import pandas as pd

from table2text.evaluation import (
    default_paths,
    generate_reports_for_notebook,
    score_reference_metrics_for_notebook,
)
from table2text.evaluation.datasets import read_examples, write_jsonl

project_dir = Path("/Users/realgobs/Documents/MScproject/table2text_pydanticai")
paths = default_paths(project_dir)

# ---- choose run size here ----
DATASET_ID = "sportsett_basketball"
N_EXAMPLES = 1
RANDOM_SAMPLE = False
SEED = 42

# Optional AlignScore setup.
# Leave as None if you have not created the separate AlignScore venv yet.
ALIGNSCORE_PYTHON = None
# Example:
# ALIGNSCORE_PYTHON = project_dir / ".venv-alignscore/bin/python"

# ---- select examples ----
examples = [
    example
    for example in read_examples(paths["prepared_examples"])
    if example.dataset_id == DATASET_ID
]

if not examples:
    raise ValueError(
        f"No prepared examples found for {DATASET_ID}. "
        "Run prepare_examples_for_notebook(project_dir) first."
    )

if RANDOM_SAMPLE:
    rng = random.Random(SEED)
    selected_examples = rng.sample(examples, min(N_EXAMPLES, len(examples)))
else:
    selected_examples = examples[:N_EXAMPLES]

run_name = f"{DATASET_ID}_{len(selected_examples)}"

examples_path = project_dir / f"evaluation/prepared/{run_name}.jsonl"
generations_path = project_dir / f"evaluation/generations/{run_name}_generations.jsonl"
run_root = project_dir / f"evaluation/generations/{run_name}_runs"
variants_path = project_dir / f"evaluation/config/variants_{run_name}.json"
metrics_path = project_dir / f"evaluation/config/metrics_{run_name}.json"
metrics_output_path = project_dir / f"evaluation/results/{run_name}_reference_metrics.jsonl"

write_jsonl(examples_path, selected_examples)

print("Selected examples:", len(selected_examples))
print("Example IDs:", [example.example_id for example in selected_examples])

# ---- use only full_system variant ----
variants_payload = json.loads(paths["variant_config"].read_text(encoding="utf-8"))
smoke_variants = {
    "variants": [
        {**variant, "enabled": variant["variant_id"] == "full_system"}
        for variant in variants_payload["variants"]
    ]
}
variants_path.write_text(json.dumps(smoke_variants, indent=2), encoding="utf-8")

# ---- metric config: BLEU/ROUGE/etc + PARENT + HHEM + AlignScore ----
metric_payload = json.loads(paths["metric_config"].read_text(encoding="utf-8"))

metric_payload["reference_metrics"]["enabled_metrics"] = [
    "bleu",
    "chrf",
    "ter",
    "rouge1",
    "rouge2",
    "rougeL",
    "rougeLsum",
    "meteor",
    "bertscore",
    #"parent",
    #"hhem",
    #"alignscore",
]

metric_payload["reference_metrics"]["hhem_model"] = (
    "vectara/hallucination_evaluation_model"
)
metric_payload["reference_metrics"]["hhem_threshold"] = 0.5
metric_payload["reference_metrics"]["hhem_batch_size"] = 16
metric_payload["reference_metrics"]["hhem_device"] = None

# metric_payload["reference_metrics"]["alignscore_python_executable"] = (
#     str(ALIGNSCORE_PYTHON) if ALIGNSCORE_PYTHON else None
# )
# metric_payload["reference_metrics"]["alignscore_model_size"] = "base"
# metric_payload["reference_metrics"]["alignscore_device"] = "cpu"
# metric_payload["reference_metrics"]["alignscore_threshold"] = 0.5

metrics_path.write_text(json.dumps(metric_payload, indent=2), encoding="utf-8")

# ---- generate reports ----
generations = await generate_reports_for_notebook(
    project_dir,
    examples_path=examples_path,
    variants_path=variants_path,
    output_path=generations_path,
    run_root=run_root,
    resume=False,
)

display(
    generations[
        [
            "dataset_id",
            "example_id",
            "variant_id",
            "error",
            "release_status",
            "writer_mode",
            "elapsed_seconds",
            "pipeline_result_path",
        ]
    ]
)

# ---- score reference + local factuality metrics ----
scores = score_reference_metrics_for_notebook(
    project_dir,
    generations_path=generations_path,
    metric_config_path=metrics_path,
    output_path=metrics_output_path,
    include_ineligible=True,
)

# ---- output vs reference table ----
references_by_id = {
    example.example_id: example.references
    for example in selected_examples
}

comparison = generations.copy()
comparison["reference"] = comparison["example_id"].map(
    lambda example_id: references_by_id.get(example_id, [""])[0]
)
comparison["all_references"] = comparison["example_id"].map(
    lambda example_id: references_by_id.get(example_id, [])
)

comparison_table = comparison[
    [
        "dataset_id",
        "example_id",
        "variant_id",
        "generated_text",
        "reference",
        "all_references",
        "release_status",
        "writer_mode",
        "pipeline_result_path",
    ]
]

display(comparison_table)

# ---- metric score table ----
score_table = scores.pivot_table(
    index=["dataset_id", "example_id", "variant_id"],
    columns="metric_name",
    values="score",
    aggfunc="first",
).reset_index()

display(score_table)

# ---- metric status table, useful for HHEM / AlignScore availability ----
status_table = scores.pivot_table(
    index=["dataset_id", "example_id", "variant_id"],
    columns="metric_name",
    values="status",
    aggfunc="first",
).reset_index()

display(status_table)

print("Examples:", examples_path)
print("Generations:", generations_path)
print("Run artifacts:", run_root)
print("Metric rows:", metrics_output_path)

In [ ]:
import json
from pathlib import Path

from table2text.evaluation import (
    default_paths,
    score_reference_metrics_for_notebook,
)

project_dir = Path("/Users/realgobs/Documents/MScproject/table2text_pydanticai")
paths = default_paths(project_dir)

generations_path = project_dir / "evaluation/generations/sportsett_basketball_1_generations.jsonl"
metrics_path = project_dir / "evaluation/config/archive/metrics_sportsett_basketball_1.json"
metrics_output_path = project_dir / "evaluation/results/sportsett_basketball_1_reference_metrics.jsonl"

metric_payload = json.loads(paths["metric_config"].read_text(encoding="utf-8"))

metric_payload["reference_metrics"]["enabled_metrics"] = [
    "bleu",
    "chrf",
    "ter",
    "rouge1",
    "rouge2",
    "rougeL",
    "rougeLsum",
    "meteor",
    "bertscore",
    "parent",
    "hhem",
    "alignscore",
]

# Leave AlignScore disabled/unavailable unless you have the separate venv.
metric_payload["reference_metrics"]["alignscore_python_executable"] = None

metrics_path.write_text(json.dumps(metric_payload, indent=2), encoding="utf-8")

scores = score_reference_metrics_for_notebook(
    project_dir,
    generations_path=generations_path,
    metric_config_path=metrics_path,
    output_path=metrics_output_path,
    include_ineligible=True,
)

display(
    scores[
        [
            "dataset_id",
            "example_id",
            "variant_id",
            "metric_family",
            "metric_name",
            "status",
            "score",
            "error",
        ]
    ]
)

score_table = scores.pivot_table(
    index=["dataset_id", "example_id", "variant_id"],
    columns="metric_name",
    values="score",
    aggfunc="first",
).reset_index()

display(score_table)

status_table = scores.pivot_table(
    index=["dataset_id", "example_id", "variant_id"],
    columns="metric_name",
    values="status",
    aggfunc="first",
).reset_index()

display(status_table)

print("Metric rows:", metrics_output_path)

In [ ]:
from pathlib import Path
from table2text.evaluation import score_deepeval_for_notebook

project_dir = Path("/Users/realgobs/Documents/MScproject/table2text_pydanticai")

deepeval_scores = score_deepeval_for_notebook(
    project_dir,
    generations_path=project_dir / "evaluation/generations/sportsett_basketball_1_generations.jsonl",
    metric_config_path=project_dir / "evaluation/config/archive/metrics_sportsett_basketball_1.json",
    output_path=project_dir / "evaluation/results/sportsett_basketball_1_deepeval_metrics.jsonl",
    resume=False,
)

deepeval_scores[
    ["dataset_id", "example_id", "metric_name", "status", "score", "reason", "error"]
]

In [ ]:
from pathlib import Path
import pandas as pd
import json

path = Path("/Users/realgobs/Documents/MScproject/table2text_pydanticai/evaluation/results/sportsett_basketball_1_deepeval_metrics.jsonl")

rows = [
    json.loads(line)
    for line in path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

df = pd.DataFrame(rows)
df[["dataset_id", "example_id", "metric_name", "judge_model", "status", "score", "success", "reason", "error"]]

In [ ]:
import json
from pathlib import Path
import pandas as pd

from table2text.evaluation import (
    default_paths,
    score_reference_metrics_for_notebook,
    score_deepeval_for_notebook,
)
from table2text.evaluation.generation import read_generations
from table2text.evaluation.datasets import write_jsonl

project_dir = Path("/Users/realgobs/Documents/MScproject/table2text_pydanticai")
paths = default_paths(project_dir)

generations_path = project_dir / "evaluation/generations/sportsett_basketball_1_generations.jsonl"

records = [
    record for record in read_generations(generations_path)
    if record.error is None and record.generated_text.strip()
]

reference_records = []
for record in records:
    for i, reference in enumerate(record.references):
        reference_records.append(
            record.model_copy(
                update={
                    "generation_id": f"{record.generation_id}::human_reference::{i}",
                    "variant_id": f"human_reference_{i + 1}",
                    "generated_text": reference,
                    "writer_mode": "provided_reference",
                    "release_status": "reference",
                    "primary_evaluation_eligible": True,
                    "primary_evaluation_reason": None,
                }
            )
        )

combined_path = project_dir / "evaluation/generations/source_grounded_generated_and_references.jsonl"
write_jsonl(combined_path, records + reference_records)

base_metrics = json.loads(paths["metric_config"].read_text(encoding="utf-8"))

source_metric_config = base_metrics.copy()
source_metric_config["reference_metrics"] = {
    **source_metric_config["reference_metrics"],
    "enabled_metrics": ["hhem", "alignscore"],
    "external_factuality_context": "source_text",
    "external_context_max_characters": 100000,
    "hhem_context_max_characters": 1500,
}
source_metric_config["deepeval"] = {
    **source_metric_config["deepeval"],
    "run_summarization": False,
    "run_reference_adequacy": False,
    "run_faithfulness": True,
    "run_factual_correctness": True,
    "run_task_relevance": True,
    "run_coherence": True,
    "run_usefulness": True,
}

source_config_path = project_dir / "evaluation/config/metrics_source_grounded.json"
source_config_path.write_text(json.dumps(source_metric_config, indent=2), encoding="utf-8")

source_local_scores = score_reference_metrics_for_notebook(
    project_dir,
    generations_path=combined_path,
    metric_config_path=source_config_path,
    output_path=project_dir / "evaluation/results/source_grounded_local_metrics.jsonl",
)

source_judge_scores = score_deepeval_for_notebook(
    project_dir,
    generations_path=combined_path,
    metric_config_path=source_config_path,
    output_path=project_dir / "evaluation/results/source_grounded_deepeval_metrics.jsonl",
    resume=False,
)

reference_similarity_config = base_metrics.copy()
reference_similarity_config["reference_metrics"] = {
    **reference_similarity_config["reference_metrics"],
    "enabled_metrics": [
        "bleu",
        "chrf",
        "ter",
        "rouge1",
        "rouge2",
        "rougeL",
        "rougeLsum",
        "meteor",
        "bertscore",
        "parent",
    ],
}

reference_config_path = project_dir / "evaluation/config/metrics_reference_similarity.json"
reference_config_path.write_text(json.dumps(reference_similarity_config, indent=2), encoding="utf-8")

reference_similarity_scores = score_reference_metrics_for_notebook(
    project_dir,
    generations_path=generations_path,
    metric_config_path=reference_config_path,
    output_path=project_dir / "evaluation/results/reference_similarity_metrics.jsonl",
)

source_scores = pd.concat(
    [
        source_local_scores.assign(metric_source="local_source_grounded"),
        source_judge_scores.assign(metric_source="deepeval_source_grounded"),
    ],
    ignore_index=True,
)

source_scores["candidate_type"] = source_scores["variant_id"].apply(
    lambda value: "provided_reference" if str(value).startswith("human_reference") else "generated"
)

source_comparison = (
    source_scores[source_scores["status"].eq("scored")]
    .groupby(["dataset_id", "example_id", "metric_name", "candidate_type"], as_index=False)["score"]
    .mean()
    .pivot_table(
        index=["dataset_id", "example_id", "metric_name"],
        columns="candidate_type",
        values="score",
        aggfunc="mean",
    )
    .reset_index()
)

if {"generated", "provided_reference"}.issubset(source_comparison.columns):
    source_comparison["generated_minus_reference"] = (
        source_comparison["generated"] - source_comparison["provided_reference"]
    )

display(source_comparison)
display(reference_similarity_scores[["dataset_id", "example_id", "variant_id", "metric_name", "status", "score"]])

In [ ]:
import json
from pathlib import Path

from table2text.evaluation import (
    default_paths,
    generate_reports_for_notebook,
    score_reference_metrics_for_notebook,
    score_deepeval_for_notebook,
)
from table2text.evaluation.datasets import read_examples, write_jsonl
from table2text.evaluation.generation import read_generations

project_dir = Path("/Users/realgobs/Documents/MScproject/table2text_pydanticai")
paths = default_paths(project_dir)

dataset_id = "sportsett_basketball"
example_id = "4934"

# 1. Select one SportSett example
examples = read_examples(paths["prepared_examples"])
example = next(
    e for e in examples
    if e.dataset_id == dataset_id and e.example_id == example_id
)

examples_path = project_dir / "evaluation/prepared/sportsett_raw_baseline_one.jsonl"
write_jsonl(examples_path, [example])

# 2. Generate raw DeepSeek V4-Pro baseline
variants_path = project_dir / "evaluation/config/variants_raw_deepseek_v4_flash.json"
variants_path.write_text(
    json.dumps(
        {
            "variants": [
                {
                    "variant_id": "raw_deepseek_v4_flash",
                    "enabled": True,
                    "backend": "callable",
                    "description": "Raw single-LLM DeepSeek baseline from source data only.",
                    "settings_overrides": {
                        "raw_baseline_model": "deepseek-v4-flash",
                        "raw_baseline_max_source_characters": 100000,
                        "raw_baseline_max_output_tokens": 3000,
                        "raw_baseline_temperature": 0.2,
                    },
                    "callable_path": "table2text.evaluation_backends.single_agent_baseline",
                    "command": [],
                    "precomputed_path": None,
                    "repetitions": 1,
                    "seeds": [42],
                }
            ]
        },
        indent=2,
    ),
    encoding="utf-8",
)

raw_generations_path = project_dir / "evaluation/generations/sportsett_raw_deepseek_v4_flash_generations.jsonl"
raw_run_root = project_dir / "evaluation/generations/sportsett_raw_deepseek_v4_flash_runs"

raw_generations = await generate_reports_for_notebook(
    project_dir,
    examples_path=examples_path,
    variants_path=variants_path,
    output_path=raw_generations_path,
    run_root=raw_run_root,
    resume=False,
)

display(raw_generations[
    ["dataset_id", "example_id", "variant_id", "generated_text", "error", "elapsed_seconds"]
])

# 3. Build comparison files: full system + raw baseline + human references
full_path = project_dir / "evaluation/generations/sportsett_basketball_1_generations.jsonl"

full_records = read_generations(full_path)
raw_records = read_generations(raw_generations_path)
system_records = full_records + raw_records

base = full_records[0]
reference_records = []
for index, reference in enumerate(base.references):
    reference_records.append(
        base.model_copy(
            update={
                "generation_id": f"{base.generation_id}::human_reference::{index}",
                "variant_id": f"human_reference_{index + 1}",
                "generated_text": reference,
                "backend": "precomputed",
                "writer_mode": "provided_reference",
                "release_status": "reference",
                "approved_for_release": True,
                "primary_evaluation_eligible": True,
                "primary_evaluation_reason": None,
                "metadata": {
                    "baseline_type": "provided_reference",
                    "reference_index": index,
                },
                "error": None,
            }
        )
    )

source_comparison_path = project_dir / "evaluation/generations/sportsett_4934_source_grounded_comparison.jsonl"
reference_similarity_path = project_dir / "evaluation/generations/sportsett_4934_reference_similarity_candidates.jsonl"

write_jsonl(source_comparison_path, system_records + reference_records)
write_jsonl(reference_similarity_path, system_records)

# 4. Create metric configs
base_metrics = json.loads(paths["metric_config"].read_text(encoding="utf-8"))

source_metric_config = base_metrics.copy()
source_metric_config["experiment_id"] = "sportsett_4934_source_grounded_comparison"
source_metric_config["reference_metrics"] = {
    **source_metric_config["reference_metrics"],
    "enabled_metrics": ["hhem", "alignscore"],
    "external_factuality_context": "source_text",
    "external_context_max_characters": 100000,
    "hhem_context_max_characters": 1500,
}
source_metric_config["deepeval"] = {
    **source_metric_config["deepeval"],
    "run_summarization": False,
    "run_reference_adequacy": False,
    "run_faithfulness": True,
    "run_factual_correctness": True,
    "run_task_relevance": True,
    "run_coherence": True,
    "run_usefulness": True,
}

source_config_path = project_dir / "evaluation/config/archive/metrics_sportsett_4934_source_grounded_comparison.json"
source_config_path.write_text(json.dumps(source_metric_config, indent=2), encoding="utf-8")

reference_metric_config = base_metrics.copy()
reference_metric_config["experiment_id"] = "sportsett_4934_reference_similarity_comparison"
reference_metric_config["reference_metrics"] = {
    **reference_metric_config["reference_metrics"],
    "enabled_metrics": [
        "bleu",
        "chrf",
        "ter",
        "rouge1",
        "rouge2",
        "rougeL",
        "rougeLsum",
        "meteor",
        "bertscore",
        "parent",
    ],
}

reference_config_path = project_dir / "evaluation/config/archive/metrics_sportsett_4934_reference_similarity_comparison.json"
reference_config_path.write_text(json.dumps(reference_metric_config, indent=2), encoding="utf-8")

# 5. Run local source-grounded metrics
source_local_scores = score_reference_metrics_for_notebook(
    project_dir,
    generations_path=source_comparison_path,
    metric_config_path=source_config_path,
    output_path=project_dir / "evaluation/results/sportsett_4934_source_grounded_local_metrics.jsonl",
)

display(source_local_scores[
    ["variant_id", "metric_name", "status", "score"]
])

# 6. Run reference-similarity metrics
reference_scores = score_reference_metrics_for_notebook(
    project_dir,
    generations_path=reference_similarity_path,
    metric_config_path=reference_config_path,
    output_path=project_dir / "evaluation/results/sportsett_4934_reference_similarity_metrics.jsonl",
)

display(reference_scores[
    ["variant_id", "metric_name", "status", "score"]
])

# 7. Optional: DeepEval source-grounded judge metrics
# This uses your DEEPSEEK_API_KEY and may take a few minutes.
deepeval_scores = score_deepeval_for_notebook(
    project_dir,
    generations_path=source_comparison_path,
    metric_config_path=source_config_path,
    output_path=project_dir / "evaluation/results/sportsett_4934_source_grounded_deepeval_metrics.jsonl",
    resume=False,
)

display(deepeval_scores[
    ["variant_id", "metric_name", "status", "score", "success"]
])

## Multi-dataset comparisons


In [ ]:
import json
import re
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from table2text.evaluation import (
    default_paths,
    generate_reports_for_notebook,
    score_reference_metrics_for_notebook,
    score_deepeval_for_notebook,
)
from table2text.evaluation.datasets import read_examples, write_jsonl

# =========================
# Config
# =========================

project_dir = Path("/Users/realgobs/Documents/MScproject/table2text_pydanticai")
paths = default_paths(project_dir)

DATASETS = [
    "dart",
]

START_INDEX = 0
EXAMPLES_PER_DATASET = 5

RAW_BASELINE_MODEL = "deepseek-v4-flash"

RUN_REFERENCE_METRICS = True
RUN_SOURCE_GROUNDED_METRICS = True
RUN_DEEPEVAL = True   # Set False if you want a faster run first.

DEEPEVAL_REPETITIONS = 1
DEEPEVAL_JUDGE_MODEL = "deepseek-v4-pro"

run_tag = datetime.now().strftime("%Y%m%d_%H%M%S")
experiment_name = f"five_dataset_five_each_comparison_{run_tag}"

# =========================
# Helpers
# =========================

def log(message):
    now = datetime.now().strftime("%H:%M:%S")
    print(f"[{now}] {message}", flush=True)

def safe_name(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value))

def write_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_generation_records(path, records):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")

def show_report(title, text):
    display(Markdown(f"## {title}\n\n{text if str(text).strip() else '*No generated text.*'}"))

def compact_metric_table(scores):
    if scores.empty:
        return pd.DataFrame()

    return (
        scores[scores["status"].isin(["scored", "error", "skipped", "unavailable"])]
        .pivot_table(
            index=["dataset_id", "example_id", "metric_name"],
            columns="variant_id",
            values="score",
            aggfunc="first",
        )
        .reset_index()
    )

# =========================
# Build variants
# =========================

default_variants = load_json(paths["variant_config"])["variants"]

full_system_variant = next(
    variant for variant in default_variants
    if variant["variant_id"] == "full_system"
)

full_system_variant = {
    **full_system_variant,
    "enabled": True,
}

raw_variant = {
    "variant_id": f"raw_{RAW_BASELINE_MODEL.replace('-', '_')}",
    "enabled": True,
    "backend": "callable",
    "description": "Raw single-LLM DeepSeek baseline from source data only.",
    "settings_overrides": {
        "raw_baseline_model": RAW_BASELINE_MODEL,
        "raw_baseline_max_source_characters": 100000,
        "raw_baseline_max_output_tokens": 3000,
        "raw_baseline_temperature": 0.2,
    },
    "callable_path": "table2text.evaluation_backends.single_agent_baseline",
    "command": [],
    "precomputed_path": None,
    "repetitions": 1,
    "seeds": [42],
}

# =========================
# Metric configs
# =========================

base_metrics = load_json(paths["metric_config"])

reference_metrics_payload = base_metrics | {
    "baseline_variant": raw_variant["variant_id"],
}

reference_metrics_payload["deepeval"] = {
    **reference_metrics_payload["deepeval"],
    "judge_model": DEEPEVAL_JUDGE_MODEL,
    "judge_repetitions": DEEPEVAL_REPETITIONS,
}

source_metrics_payload = json.loads(json.dumps(reference_metrics_payload))
source_metrics_payload["reference_metrics"]["external_factuality_context"] = "source_text"
source_metrics_payload["reference_metrics"]["enabled_metrics"] = [
    "hhem",
    "alignscore",
]
source_metrics_payload["deepeval"] = {
    **source_metrics_payload["deepeval"],
    "run_summarization": False,
    "run_reference_adequacy": False,
}

reference_metrics_path = project_dir / f"evaluation/config/metrics_{experiment_name}_reference.json"
source_metrics_path = project_dir / f"evaluation/config/metrics_{experiment_name}_source_grounded.json"

write_json(reference_metrics_path, reference_metrics_payload)
write_json(source_metrics_path, source_metrics_payload)

# =========================
# Select 5 examples each
# =========================

all_examples = read_examples(paths["prepared_examples"])

selected_by_dataset = {}

for dataset_id in DATASETS:
    dataset_examples = [e for e in all_examples if e.dataset_id == dataset_id]

    if not dataset_examples:
        raise ValueError(f"No prepared examples found for dataset: {dataset_id}")

    selected = dataset_examples[START_INDEX : START_INDEX + EXAMPLES_PER_DATASET]

    if len(selected) < EXAMPLES_PER_DATASET:
        raise ValueError(
            f"Only found {len(selected)} examples for {dataset_id}, "
            f"but requested {EXAMPLES_PER_DATASET}."
        )

    selected_by_dataset[dataset_id] = selected

log(f"Experiment: {experiment_name}")
log(f"Datasets: {DATASETS}")
log(f"Examples per dataset: {EXAMPLES_PER_DATASET}")
log(f"Raw baseline: {RAW_BASELINE_MODEL}")
log(f"DeepEval: {RUN_DEEPEVAL}, judge={DEEPEVAL_JUDGE_MODEL}, reps={DEEPEVAL_REPETITIONS}")

# =========================
# Run dataset-by-dataset
# =========================

all_generation_records = []
all_reference_scores = []
all_source_scores = []
all_deepeval_scores = []

for dataset_number, dataset_id in enumerate(DATASETS, start=1):
    examples = selected_by_dataset[dataset_id]

    log("=" * 80)
    log(f"DATASET {dataset_number}/{len(DATASETS)}: {dataset_id}")
    log(f"Selected examples: {[e.example_id for e in examples]}")

    examples_path = project_dir / f"evaluation/prepared/{experiment_name}_{dataset_id}_five.jsonl"
    write_jsonl(examples_path, examples)

    dataset_generation_records = []

    for variant in [full_system_variant, raw_variant]:
        variant_id = variant["variant_id"]

        log("-" * 80)
        log(f"Starting generation: dataset={dataset_id}, variant={variant_id}, count={len(examples)}")

        single_variant_path = (
            project_dir
            / f"evaluation/config/variants_{experiment_name}_{dataset_id}_{variant_id}.json"
        )
        write_json(single_variant_path, {"variants": [variant]})

        variant_generations_path = (
            project_dir
            / f"evaluation/generations/{experiment_name}_{dataset_id}_{variant_id}_generations.jsonl"
        )
        run_root = (
            project_dir
            / f"evaluation/generations/{experiment_name}_runs/{dataset_id}/{variant_id}"
        )

        t0 = time.perf_counter()

        generation_frame = await generate_reports_for_notebook(
            project_dir,
            examples_path=examples_path,
            variants_path=single_variant_path,
            output_path=variant_generations_path,
            run_root=run_root,
            resume=False,
        )

        elapsed = time.perf_counter() - t0
        records = generation_frame.to_dict("records")

        dataset_generation_records.extend(records)
        all_generation_records.extend(records)

        log(
            f"Finished generation: dataset={dataset_id}, variant={variant_id}, "
            f"elapsed={elapsed:.1f}s, rows={len(records)}"
        )

        display(
            generation_frame[
                [
                    "dataset_id",
                    "example_id",
                    "variant_id",
                    "error",
                    "release_status",
                    "writer_mode",
                    "elapsed_seconds",
                ]
            ]
        )

        for row in records:
            log(
                f"Generated report: dataset={row.get('dataset_id')}, "
                f"example={row.get('example_id')}, variant={row.get('variant_id')}, "
                f"error={row.get('error')}"
            )
            show_report(
                f"{row.get('dataset_id')} / {row.get('example_id')} / {row.get('variant_id')}",
                row.get("generated_text") or "",
            )

    # Combined generation file for this dataset.
    dataset_generations_path = (
        project_dir
        / f"evaluation/generations/{experiment_name}_{dataset_id}_combined_generations.jsonl"
    )
    write_generation_records(dataset_generations_path, dataset_generation_records)

    log(f"Wrote combined dataset generations: {dataset_generations_path}")

    # =========================
    # Metrics for this dataset
    # =========================

    if RUN_REFERENCE_METRICS:
        log(f"Starting reference metrics for {dataset_id}")

        reference_scores_path = (
            project_dir
            / f"evaluation/results/{experiment_name}_{dataset_id}_reference_metrics.jsonl"
        )

        t0 = time.perf_counter()
        reference_scores = score_reference_metrics_for_notebook(
            project_dir,
            generations_path=dataset_generations_path,
            metric_config_path=reference_metrics_path,
            output_path=reference_scores_path,
            include_ineligible=True,
        )
        log(f"Finished reference metrics for {dataset_id} in {time.perf_counter() - t0:.1f}s")

        all_reference_scores.append(reference_scores)

        print(f"\nREFERENCE METRICS: {dataset_id}")
        display(compact_metric_table(reference_scores))

    if RUN_SOURCE_GROUNDED_METRICS:
        log(f"Starting source-grounded HHEM/AlignScore metrics for {dataset_id}")

        source_scores_path = (
            project_dir
            / f"evaluation/results/{experiment_name}_{dataset_id}_source_grounded_metrics.jsonl"
        )

        t0 = time.perf_counter()
        source_scores = score_reference_metrics_for_notebook(
            project_dir,
            generations_path=dataset_generations_path,
            metric_config_path=source_metrics_path,
            output_path=source_scores_path,
            include_ineligible=True,
        )
        log(f"Finished source-grounded metrics for {dataset_id} in {time.perf_counter() - t0:.1f}s")

        all_source_scores.append(source_scores)

        print(f"\nSOURCE-GROUNDED METRICS: {dataset_id}")
        display(compact_metric_table(source_scores))

    if RUN_DEEPEVAL:
        log(f"Starting DeepEval metrics for {dataset_id}. This can be slow.")

        deepeval_scores_path = (
            project_dir
            / f"evaluation/results/{experiment_name}_{dataset_id}_deepeval_metrics.jsonl"
        )

        t0 = time.perf_counter()
        deepeval_scores = score_deepeval_for_notebook(
            project_dir,
            generations_path=dataset_generations_path,
            metric_config_path=source_metrics_path,
            output_path=deepeval_scores_path,
            resume=False,
        )
        log(f"Finished DeepEval metrics for {dataset_id} in {time.perf_counter() - t0:.1f}s")

        all_deepeval_scores.append(deepeval_scores)

        print(f"\nDEEPEVAL METRICS: {dataset_id}")
        display(compact_metric_table(deepeval_scores))

# =========================
# Write combined files
# =========================

combined_generations_path = (
    project_dir
    / f"evaluation/generations/{experiment_name}_ALL_generations.jsonl"
)
write_generation_records(combined_generations_path, all_generation_records)

log("=" * 80)
log("ALL DATASETS COMPLETE")
log(f"Combined generations: {combined_generations_path}")
log(f"Reference metrics config: {reference_metrics_path}")
log(f"Source-grounded metrics config: {source_metrics_path}")

generations_all = pd.DataFrame(all_generation_records)

reference_all = (
    pd.concat(all_reference_scores, ignore_index=True)
    if all_reference_scores
    else pd.DataFrame()
)

source_all = (
    pd.concat(all_source_scores, ignore_index=True)
    if all_source_scores
    else pd.DataFrame()
)

deepeval_all = (
    pd.concat(all_deepeval_scores, ignore_index=True)
    if all_deepeval_scores
    else pd.DataFrame()
)

print("\nALL GENERATIONS")
display(
    generations_all[
        [
            "dataset_id",
            "example_id",
            "variant_id",
            "error",
            "release_status",
            "writer_mode",
            "elapsed_seconds",
        ]
    ]
)

print("\nALL REFERENCE METRICS")
display(compact_metric_table(reference_all))

print("\nALL SOURCE-GROUNDED METRICS")
display(compact_metric_table(source_all))

print("\nALL DEEPEVAL METRICS")
display(compact_metric_table(deepeval_all))

In [ ]:
import json
import os
from pathlib import Path

from table2text.evaluation import (
    default_paths,
    generate_reports_for_notebook,
    score_reference_metrics_for_notebook,
    load_project_env,
)
from table2text.evaluation.datasets import read_examples, write_jsonl

project_dir = Path("/Users/realgobs/Documents/MScproject/table2text_pydanticai")
paths = default_paths(project_dir)
load_project_env(project_dir)

dataset_id = "sportsett_basketball"
example_id = "4934"

examples = read_examples(paths["prepared_examples"])
one_example = next(
    e for e in examples
    if e.dataset_id == dataset_id and str(e.example_id) == example_id
)

examples_path = project_dir / f"evaluation/prepared/{dataset_id}_{example_id}_fast_compare.jsonl"
write_jsonl(examples_path, [one_example])

variants = {
    "variants": [
        {
            "variant_id": "full_system_fast",
            "enabled": True,
            "backend": "table2text",
            "description": "Fast full-system run using .env flash routing.",
            "settings_overrides": {
                "use_llm": True,
                "structured_output_mode": "prompted",
                "max_total_tokens": 300000,
                "max_agent_requests": 5,
                "max_revision_rounds": 0,
                "writer_quality_revision_rounds": 0,
                "enable_insight_synthesis": True,
            },
            "callable_path": None,
            "command": [],
            "precomputed_path": None,
            "repetitions": 1,
            "seeds": [42],
        },
        {
            "variant_id": "raw_deepseek_v4_pro",
            "enabled": True,
            "backend": "callable",
            "description": "Raw one-shot DeepSeek v4-pro baseline.",
            "settings_overrides": {
                "raw_baseline_model": "deepseek-v4-pro",
                "raw_baseline_max_source_characters": 100000,
                "raw_baseline_max_output_tokens": 3000,
                "raw_baseline_temperature": 0.2,
            },
            "callable_path": "table2text.evaluation_backends.single_agent_baseline",
            "command": [],
            "precomputed_path": None,
            "repetitions": 1,
            "seeds": [42],
        },
    ]
}

variants_path = project_dir / f"evaluation/config/variants_{dataset_id}_{example_id}_fast_compare.json"
variants_path.write_text(json.dumps(variants, indent=2), encoding="utf-8")

generations_path = project_dir / f"evaluation/generations/{dataset_id}_{example_id}_fast_compare_generations.jsonl"
run_root = project_dir / f"evaluation/generations/{dataset_id}_{example_id}_fast_compare_runs"

print("Model routing from .env:")
for key in [
    "T2T_MODEL_DATA_UNDERSTANDING",
    "T2T_MODEL_ORCHESTRATOR",
    "T2T_MODEL_EVIDENCE",
    "T2T_MODEL_VERIFIER",
    "T2T_MODEL_WRITER",
    "T2T_MODEL_AUDITOR",
]:
    print(f"{key} = {os.getenv(key)}")

print("\nGenerating full system + raw baseline...")
generations = await generate_reports_for_notebook(
    project_dir,
    examples_path=examples_path,
    variants_path=variants_path,
    output_path=generations_path,
    run_root=run_root,
    resume=False,
)

for _, row in generations.iterrows():
    print("\n" + "=" * 80)
    print("Variant:", row["variant_id"])
    print("Error:", row["error"])
    print("Release status:", row.get("release_status"))
    print("Writer mode:", row.get("writer_mode"))
    print("Elapsed seconds:", row.get("elapsed_seconds"))
    print("Artifact:", row.get("pipeline_result_path"))
    print("-" * 80)
    print(row["generated_text"])

# Reference metrics only. No DeepEval.
metrics_path = project_dir / f"evaluation/config/metrics_{dataset_id}_{example_id}_fast_reference_only.json"

metrics_config = {
    "experiment_id": f"{dataset_id}_{example_id}_fast_reference_only",
    "baseline_variant": "raw_deepseek_v4_pro",
    "reference_metrics": {
        "run_bleu": True,
        "run_chrf": True,
        "run_ter": True,
        "run_rouge": True,
        "run_meteor": True,
        "run_bertscore": True,
        "run_parent": True,
        "run_hhem": False,
        "run_alignscore": False,
    },
    "deepeval": {
        "enabled": False,
    },
}
metrics_path.write_text(json.dumps(metrics_config, indent=2), encoding="utf-8")

print("\nScoring reference metrics...")
scores_path = project_dir / f"evaluation/results/{dataset_id}_{example_id}_fast_reference_metrics.jsonl"

scores = score_reference_metrics_for_notebook(
    project_dir,
    generations_path=generations_path,
    metric_config_path=metrics_path,
    output_path=scores_path,
)

print("\nMetric comparison:")
comparison = (
    scores[
        scores["status"].eq("scored")
        & scores["variant_id"].isin(["full_system_fast", "raw_deepseek_v4_flash"])
    ]
    .pivot_table(
        index="metric_name",
        columns="variant_id",
        values="score",
        aggfunc="mean",
    )
    .reset_index()
)

if {"full_system_fast", "raw_deepseek_v4_flash"}.issubset(comparison.columns):
    comparison["delta_system_minus_raw"] = (
        comparison["full_system_fast"] - comparison["raw_deepseek_v4_flash"]
    )

display(comparison)

print("\nGenerations saved to:", generations_path)
print("Metrics saved to:", scores_path)

generations[
    [
        "dataset_id",
        "example_id",
        "variant_id",
        "generated_text",
        "error",
        "release_status",
        "writer_mode",
        "elapsed_seconds",
        "pipeline_result_path",
    ]
]

In [ ]:
import json
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from table2text.evaluation import (
    default_paths,
    generate_reports_for_notebook,
    score_reference_metrics_for_notebook,
    load_project_env,
)
from table2text.evaluation.datasets import read_examples, write_jsonl

project_dir = Path("/Users/realgobs/Documents/MScproject/table2text_pydanticai")
paths = default_paths(project_dir)
load_project_env(project_dir)

DATASETS = ["e2e_nlg", "totto", "web_nlg", "dart"]
START_INDEX = 0

run_tag = datetime.now().strftime("%Y%m%d_%H%M%S")
experiment_name = f"four_dataset_pro_comparison_{run_tag}"

def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)

def write_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

def write_generation_records(path, records):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")

def show_report(title, text):
    display(Markdown(f"## {title}\n\n{text if str(text).strip() else '*No generated text.*'}"))

def compact_metric_table(scores):
    if scores.empty:
        return pd.DataFrame()

    return (
        scores[scores["status"].eq("scored")]
        .pivot_table(
            index=["dataset_id", "example_id", "metric_name"],
            columns="variant_id",
            values="score",
            aggfunc="first",
        )
        .reset_index()
    )

examples = read_examples(paths["prepared_examples"])

selected_examples = []
for dataset_id in DATASETS:
    dataset_examples = [e for e in examples if e.dataset_id == dataset_id]
    if not dataset_examples:
        raise ValueError(f"No prepared examples found for {dataset_id}")
    selected_examples.append(dataset_examples[START_INDEX])

full_system_pro = {
    "variant_id": "full_system_pro",
    "enabled": True,
    "backend": "table2text",
    "description": "Full Table2Text workflow with all agents on DeepSeek v4-pro.",
    "settings_overrides": {
        "use_llm": True,
        "structured_output_mode": "prompted",
        "max_total_tokens": 300000,
        "max_agent_requests": 8,

        "data_understanding_model": "deepseek:deepseek-v4-pro",
        "orchestrator_model": "deepseek:deepseek-v4-pro",
        "evidence_model": "deepseek:deepseek-v4-pro",
        "verifier_model": "deepseek:deepseek-v4-pro",
        "writer_model": "deepseek:deepseek-v4-pro",
        "auditor_model": "deepseek:deepseek-v4-pro",
    },
    "callable_path": None,
    "command": [],
    "precomputed_path": None,
    "repetitions": 1,
    "seeds": [42],
}

raw_generic_pro = {
    "variant_id": "raw_generic_pro",
    "enabled": True,
    "backend": "callable",
    "description": "Raw generic one-shot DeepSeek v4-pro baseline.",
    "settings_overrides": {
        "raw_baseline_model": "deepseek-v4-pro",
        "raw_baseline_prompt_mode": "generic",
        "raw_baseline_max_source_characters": 100000,
        "raw_baseline_max_output_tokens": 3000,
        "raw_baseline_temperature": 0.2,
    },
    "callable_path": "table2text.evaluation_backends.single_agent_baseline",
    "command": [],
    "precomputed_path": None,
    "repetitions": 1,
    "seeds": [42],
}

variants_path = project_dir / f"evaluation/config/variants_{experiment_name}.json"
write_json(variants_path, {"variants": [full_system_pro, raw_generic_pro]})

base_metrics = json.loads(paths["metric_config"].read_text(encoding="utf-8"))
metrics_config = {
    **base_metrics,
    "experiment_id": experiment_name,
    "prepared_examples_path": str(paths["prepared_examples"]),
    "generations_path": str(
        project_dir / f"evaluation/generations/{experiment_name}_combined_generations.jsonl"
    ),
    "baseline_variant": "raw_generic_pro",
}
metrics_config["reference_metrics"]["enabled_metrics"] = [
    "bleu",
    "chrf",
    "ter",
    "rouge1",
    "rouge2",
    "rougeL",
    "rougeLsum",
    "meteor",
    "bertscore",
]
metrics_config["deepeval"]["enabled"] = False

metrics_path = project_dir / f"evaluation/config/metrics_{experiment_name}.json"
write_json(metrics_path, metrics_config)

all_generation_records = []
all_scores = []

log(f"Experiment: {experiment_name}")
log(f"Datasets: {DATASETS}")
log("Running full_system_pro vs raw_generic_pro")

for i, example in enumerate(selected_examples, start=1):
    dataset_id = example.dataset_id
    example_id = example.example_id

    log("=" * 80)
    log(f"Dataset {i}/{len(selected_examples)}: {dataset_id} / {example_id}")

    examples_path = project_dir / f"evaluation/prepared/{experiment_name}_{dataset_id}_{example_id}.jsonl"
    write_jsonl(examples_path, [example])

    dataset_generations_path = (
        project_dir / f"evaluation/generations/{experiment_name}_{dataset_id}_generations.jsonl"
    )
    run_root = project_dir / f"evaluation/generations/{experiment_name}_runs/{dataset_id}"

    t0 = time.perf_counter()
    generations = await generate_reports_for_notebook(
        project_dir,
        examples_path=examples_path,
        variants_path=variants_path,
        output_path=dataset_generations_path,
        run_root=run_root,
        resume=False,
    )
    log(f"Generated {dataset_id} in {time.perf_counter() - t0:.1f}s")

    records = generations.to_dict("records")
    all_generation_records.extend(records)

    for _, row in generations.iterrows():
        log(
            f"{row['variant_id']}: error={row.get('error')}, "
            f"release={row.get('release_status')}, "
            f"writer={row.get('writer_mode')}, "
            f"elapsed={row.get('elapsed_seconds')}"
        )
        show_report(
            f"{dataset_id} / {example_id} / {row['variant_id']}",
            row.get("generated_text") or "",
        )

    scores_path = project_dir / f"evaluation/results/{experiment_name}_{dataset_id}_reference_metrics.jsonl"
    scores = score_reference_metrics_for_notebook(
        project_dir,
        generations_path=dataset_generations_path,
        metric_config_path=metrics_path,
        output_path=scores_path,
        include_ineligible=True,
    )
    all_scores.append(scores)

    print(f"\nMETRICS: {dataset_id}")
    display(compact_metric_table(scores))

combined_generations_path = project_dir / f"evaluation/generations/{experiment_name}_combined_generations.jsonl"
write_generation_records(combined_generations_path, all_generation_records)

combined_scores = pd.concat(all_scores, ignore_index=True) if all_scores else pd.DataFrame()
combined_scores_path = project_dir / f"evaluation/results/{experiment_name}_reference_metrics_combined.jsonl"

with combined_scores_path.open("w", encoding="utf-8") as handle:
    for record in combined_scores.to_dict("records"):
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")

print("\nALL DONE")
print("Combined generations:", combined_generations_path)
print("Combined metrics:", combined_scores_path)

print("\nGENERATION SUMMARY")
display(
    pd.DataFrame(all_generation_records)[
        [
            "dataset_id",
            "example_id",
            "variant_id",
            "error",
            "release_status",
            "writer_mode",
            "elapsed_seconds",
        ]
    ]
)

print("\nCOMBINED METRICS")
display(compact_metric_table(combined_scores))

## Focused SportSett diagnostics


In [ ]:
import json
from pathlib import Path
from table2text.evaluation import default_paths, score_reference_metrics_for_notebook

project_dir = Path("/Users/realgobs/Documents/MScproject/table2text_pydanticai")
paths = default_paths(project_dir)

dataset_id = "sportsett_basketball"
example_id = "4934"

generations_path = project_dir / f"evaluation/generations/{dataset_id}_{example_id}_fast_compare_generations.jsonl"
metrics_path = project_dir / f"evaluation/config/metrics_{dataset_id}_{example_id}_fast_reference_only_fixed.json"
scores_path = project_dir / f"evaluation/results/{dataset_id}_{example_id}_fast_reference_metrics_fixed.jsonl"

base_metrics = json.loads(paths["metric_config"].read_text(encoding="utf-8"))

metrics_config = {
    **base_metrics,
    "experiment_id": f"{dataset_id}_{example_id}_fast_reference_only_fixed",
    "prepared_examples_path": str(paths["prepared_examples"]),
    "generations_path": str(generations_path),
    "baseline_variant": "raw_deepseek_v4_pro",
}

metrics_config["reference_metrics"]["enabled_metrics"] = [
    "bleu",
    "chrf",
    "ter",
    "rouge1",
    "rouge2",
    "rougeL",
    "rougeLsum",
    "meteor",
    "bertscore",
]

metrics_config["deepeval"]["enabled"] = False

metrics_path.write_text(json.dumps(metrics_config, indent=2), encoding="utf-8")

scores = score_reference_metrics_for_notebook(
    project_dir,
    generations_path=generations_path,
    metric_config_path=metrics_path,
    output_path=scores_path,
    include_ineligible=True,
)

comparison = (
    scores[
        scores["status"].eq("scored")
        & scores["variant_id"].isin(["full_system_fast", "raw_deepseek_v4_pro"])
    ]
    .pivot_table(
        index="metric_name",
        columns="variant_id",
        values="score",
        aggfunc="mean",
    )
    .reset_index()
)

if {"full_system_fast", "raw_deepseek_v4_pro"}.issubset(comparison.columns):
    comparison["delta_system_minus_raw_pro"] = (
        comparison["full_system_fast"] - comparison["raw_deepseek_v4_pro"]
    )

display(comparison)

print("Scored existing generations from:", generations_path)
print("Metrics saved to:", scores_path)

In [ ]:
# Reference metrics only. No DeepEval.
metrics_path = project_dir / f"evaluation/config/metrics_{dataset_id}_{example_id}_fast_reference_only.json"

metrics_config = {
    "experiment_id": f"{dataset_id}_{example_id}_fast_reference_only",
    "prepared_examples_path": str(examples_path),
    "generations_path": str(generations_path),
    "result_directory": str(project_dir / "evaluation/results"),
    "baseline_variant": "raw_deepseek_v4_flash",
    "bootstrap_resamples": 5000,
    "confidence_level": 0.95,
    "random_seed": 42,
    "reference_metrics": {
        "enabled_metrics": [
            "bleu",
            "chrf",
            "ter",
            "rouge1",
            "rouge2",
            "rougeL",
            "rougeLsum",
            "meteor",
            "bertscore",
            "parent",
        ],
        "bertscore_model": "roberta-base",
        "bertscore_num_layers": None,
        "bertscore_batch_size": 8,
        "bertscore_device": None,
        "bertscore_rescale_with_baseline": False,
        "parent_n_jobs": 1,
        "hf_local_files_only": True,
        "external_factuality_context": "references",
        "lowercase": False,
    },
    "deepeval": {
        "enabled": False,
        "judge_provider": "deepseek",
        "judge_model": "deepseek-v4-pro",
        "judge_repetitions": 1,
        "threshold": 0.5,
        "max_source_characters": 50000,
        "run_summarization": False,
        "run_faithfulness": False,
        "run_factual_correctness": False,
        "run_reference_adequacy": False,
        "run_task_relevance": False,
        "run_coherence": False,
        "run_usefulness": False,
    },
}

metrics_path.write_text(json.dumps(metrics_config, indent=2), encoding="utf-8")

print("\nScoring reference metrics...")
scores_path = project_dir / f"evaluation/results/{dataset_id}_{example_id}_fast_reference_metrics.jsonl"

scores = score_reference_metrics_for_notebook(
    project_dir,
    generations_path=generations_path,
    metric_config_path=metrics_path,
    output_path=scores_path,
)

comparison = (
    scores[
        scores["status"].eq("scored")
        & scores["variant_id"].isin(["full_system_fast", "raw_deepseek_v4_flash"])
    ]
    .pivot_table(
        index="metric_name",
        columns="variant_id",
        values="score",
        aggfunc="mean",
    )
    .reset_index()
)

if {"full_system_fast", "raw_deepseek_v4_flash"}.issubset(comparison.columns):
    comparison["delta_system_minus_raw"] = (
        comparison["full_system_fast"] - comparison["raw_deepseek_v4_flash"]
    )

display(comparison)

print("Metrics saved to:", scores_path)

In [ ]:
import json
import os
from pathlib import Path

from table2text.evaluation import (
    default_paths,
    generate_reports_for_notebook,
    load_project_env,
)
from table2text.evaluation.datasets import read_examples, write_jsonl

project_dir = Path("/Users/realgobs/Documents/MScproject/table2text_pydanticai")
paths = default_paths(project_dir)
load_project_env(project_dir)

dataset_id = "sportsett_basketball"
example_id = "4934"

examples = read_examples(paths["prepared_examples"])
one_example = next(
    e for e in examples
    if e.dataset_id == dataset_id and str(e.example_id) == example_id
)

examples_path = project_dir / f"evaluation/prepared/{dataset_id}_{example_id}_raw_only.jsonl"
write_jsonl(examples_path, [one_example])

variants = {
    "variants": [
        {
            "variant_id": "raw_deepseek_v4_flash",
            "enabled": True,
            "backend": "callable",
            "description": "Raw one-shot DeepSeek flash baseline.",
            "settings_overrides": {
                "raw_baseline_model": "deepseek-v4-flash",
                "raw_baseline_max_source_characters": 100000,
                "raw_baseline_max_output_tokens": 1500,
                "raw_baseline_temperature": 0.2,
            },
            "callable_path": "table2text.evaluation_backends.single_agent_baseline",
            "command": [],
            "precomputed_path": None,
            "repetitions": 1,
            "seeds": [42],
        }
    ]
}

variants_path = project_dir / f"evaluation/config/variants_{dataset_id}_{example_id}_raw_only.json"
variants_path.write_text(json.dumps(variants, indent=2), encoding="utf-8")

generations_path = project_dir / f"evaluation/generations/{dataset_id}_{example_id}_raw_only_generations.jsonl"
run_root = project_dir / f"evaluation/generations/{dataset_id}_{example_id}_raw_only_runs"

print("Raw model:", variants["variants"][0]["settings_overrides"]["raw_baseline_model"])
print("Running raw baseline...")

raw_generations = await generate_reports_for_notebook(
    project_dir,
    examples_path=examples_path,
    variants_path=variants_path,
    output_path=generations_path,
    run_root=run_root,
    resume=False,
)

row = raw_generations.iloc[0]

print("\nDone.")
print("Variant:", row["variant_id"])
print("Error:", row["error"])
print("Elapsed seconds:", row["elapsed_seconds"])

print("\nGenerated text:\n")
print(row["generated_text"])

raw_generations[
    [
        "dataset_id",
        "example_id",
        "variant_id",
        "generated_text",
        "error",
        "elapsed_seconds",
    ]
]

In [ ]:
import json
from pathlib import Path

from table2text.evaluation import score_reference_metrics_for_notebook

def read_jsonl_file(path):
    rows = []
    with Path(path).open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def write_jsonl_file(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")

full_system_path = project_dir / "evaluation/generations/sportsett_basketball_4934_fast_compare_generations.jsonl"
raw_path = project_dir / "evaluation/generations/sportsett_basketball_4934_raw_only_generations.jsonl"

full_rows = [
    row for row in read_jsonl_file(full_system_path)
    if row.get("variant_id") == "full_system_fast"
]

raw_rows = [
    row for row in read_jsonl_file(raw_path)
    if row.get("variant_id") == "raw_deepseek_v4_flash"
]

print("Full-system rows:", len(full_rows))
print("Raw rows:", len(raw_rows))

combined_path = project_dir / "evaluation/generations/sportsett_basketball_4934_fast_vs_raw_generations.jsonl"
write_jsonl_file(combined_path, [*full_rows, *raw_rows])

metrics_path = project_dir / "evaluation/config/archive/metrics_sportsett_basketball_4934_fast_vs_raw_reference_only.json"

metrics_config = {
    "experiment_id": "sportsett_basketball_4934_fast_vs_raw_reference_only",
    "prepared_examples_path": str(project_dir / "evaluation/prepared/sportsett_basketball_4934_raw_only.jsonl"),
    "generations_path": str(combined_path),
    "result_directory": str(project_dir / "evaluation/results"),
    "baseline_variant": "raw_deepseek_v4_flash",
    "bootstrap_resamples": 5000,
    "confidence_level": 0.95,
    "random_seed": 42,
    "reference_metrics": {
        "enabled_metrics": [
            "bleu",
            "chrf",
            "ter",
            "rouge1",
            "rouge2",
            "rougeL",
            "rougeLsum",
            "meteor",
            "bertscore",
            "parent",
        ],
        "bertscore_model": "roberta-base",
        "bertscore_num_layers": None,
        "bertscore_batch_size": 8,
        "bertscore_device": None,
        "bertscore_rescale_with_baseline": False,
        "parent_n_jobs": 1,
        "hf_local_files_only": True,
        "external_factuality_context": "references",
        "lowercase": False,
    },
    "deepeval": {
        "enabled": False,
        "judge_provider": "deepseek",
        "judge_model": "deepseek-v4-pro",
        "judge_repetitions": 1,
        "threshold": 0.5,
        "max_source_characters": 50000,
        "run_summarization": False,
        "run_faithfulness": False,
        "run_factual_correctness": False,
        "run_reference_adequacy": False,
        "run_task_relevance": False,
        "run_coherence": False,
        "run_usefulness": False,
    },
}

metrics_path.write_text(json.dumps(metrics_config, indent=2), encoding="utf-8")

scores_path = project_dir / "evaluation/results/sportsett_basketball_4934_fast_vs_raw_reference_metrics.jsonl"

scores = score_reference_metrics_for_notebook(
    project_dir,
    generations_path=combined_path,
    metric_config_path=metrics_path,
    output_path=scores_path,
)

comparison = (
    scores[scores["status"].eq("scored")]
    .pivot_table(
        index="metric_name",
        columns="variant_id",
        values="score",
        aggfunc="mean",
    )
    .reset_index()
)

if {"full_system_fast", "raw_deepseek_v4_flash"}.issubset(comparison.columns):
    comparison["delta_system_minus_raw"] = (
        comparison["full_system_fast"] - comparison["raw_deepseek_v4_flash"]
    )

display(comparison)

print("Combined generations:", combined_path)
print("Metrics:", scores_path)

## Model-routing experiments


In [ ]:
import json
from pathlib import Path

from table2text.evaluation import (
    default_paths,
    generate_reports_for_notebook,
    score_reference_metrics_for_notebook,
)
from table2text.evaluation.datasets import read_examples, write_jsonl

project_dir = Path("/Users/realgobs/Documents/MScproject/table2text_pydanticai")
paths = default_paths(project_dir)

# Same five examples we have been using
TARGETS = [
    ("sportsett_basketball", "4934"),
    ("e2e_nlg", "e2e_nlg-test-51"),
    ("totto", "totto-validation-204"),
    ("web_nlg", "web_nlg_en-test-51"),
    ("dart", "dart-test-53"),
]

examples = read_examples(paths["prepared_examples"])
selected = []

for dataset_id, example_id in TARGETS:
    match = next(
        e for e in examples
        if e.dataset_id == dataset_id and e.example_id == example_id
    )
    selected.append(match)

examples_path = project_dir / "evaluation/prepared/five_dataset_writer_gpt55_examples.jsonl"
write_jsonl(examples_path, selected)

variant = {
    "variants": [
        {
            "variant_id": "full_system_flash_pipeline_gpt55_writer",
            "enabled": True,
            "backend": "table2text",
            "description": "DeepSeek flash/basic pipeline with OpenAI GPT-5.5 writer.",
            "settings_overrides": {
                "use_llm": True,
                "structured_output_mode": "prompted",
                "max_total_tokens": 80000,
                "max_agent_requests": 8,

                "force_llm_short_form_writer": False,

                "data_understanding_model": "deepseek:deepseek-v4-flash",
                "orchestrator_model": "deepseek:deepseek-v4-flash",
                "evidence_model": "deepseek:deepseek-v4-flash",
                "verifier_model": "deepseek:deepseek-v4-flash",
                "writer_model": "openai:gpt-5.5",
                "auditor_model": "deepseek:deepseek-v4-flash",
            },
            "callable_path": None,
            "command": [],
            "precomputed_path": None,
            "repetitions": 1,
            "seeds": [42],
        }
    ]
}

variants_path = project_dir / "evaluation/config/archive/variants_five_dataset_flash_pipeline_gpt55_writer.json"
variants_path.parent.mkdir(parents=True, exist_ok=True)
variants_path.write_text(json.dumps(variant, indent=2), encoding="utf-8")

generations_path = project_dir / "evaluation/generations/five_dataset_flash_pipeline_gpt55_writer_generations.jsonl"
run_root = project_dir / "evaluation/generations/five_dataset_flash_pipeline_gpt55_writer_runs"

generations = await generate_reports_for_notebook(
    project_dir,
    examples_path=examples_path,
    variants_path=variants_path,
    output_path=generations_path,
    run_root=run_root,
    resume=False,
)

display(
    generations[
        [
            "dataset_id",
            "example_id",
            "variant_id",
            "generated_text",
            "error",
            "release_status",
            "writer_mode",
            "elapsed_seconds",
        ]
    ]
)

# Reference metrics, no DeepEval here
scores_path = project_dir / "evaluation/results/five_dataset_flash_pipeline_gpt55_writer_reference_metrics.jsonl"

scores = score_reference_metrics_for_notebook(
    project_dir,
    generations_path=generations_path,
    metric_config_path=paths["metric_config"],
    output_path=scores_path,
    include_ineligible=True,
)

metric_table = (
    scores[scores["status"].eq("scored")]
    .pivot_table(
        index=["dataset_id", "example_id", "metric_name"],
        columns="variant_id",
        values="score",
        aggfunc="first",
    )
    .reset_index()
)

display(metric_table)

print("Examples:", examples_path)
print("Variants:", variants_path)
print("Generations:", generations_path)
print("Run root:", run_root)
print("Scores:", scores_path)

In [ ]:
import json
from pathlib import Path

project_dir = Path("/Users/realgobs/Documents/MScproject/table2text_pydanticai")
variant_path = project_dir / "evaluation/config/archive/variants_full_system_openai_gpt55_same_as_pro.json"

payload = json.loads(variant_path.read_text(encoding="utf-8"))
payload["variants"][0]["settings_overrides"]["force_llm_short_form_writer"]

In [ ]:
import json
import time
from datetime import datetime
from pathlib import Path

from table2text.evaluation import (
    default_paths,
    generate_reports_for_notebook,
    score_reference_metrics_for_notebook,
    load_project_env,
)
from table2text.evaluation.datasets import read_examples, write_jsonl

project_dir = Path("/Users/realgobs/Documents/MScproject/table2text_pydanticai")
paths = default_paths(project_dir)
load_project_env(project_dir)

dataset_id = "sportsett_basketball"
example_id = "4934"

# Change this if you want pro.
MODEL = "deepseek-v4-flash"
# MODEL = "deepseek-v4-pro"

run_tag = datetime.now().strftime("%Y%m%d_%H%M%S")
experiment_name = f"{dataset_id}_{example_id}_inferred_contract_{MODEL}_{run_tag}"

examples = read_examples(paths["prepared_examples"])
one_example = next(
    e for e in examples
    if e.dataset_id == dataset_id and str(e.example_id) == example_id
)

examples_path = project_dir / f"evaluation/prepared/{experiment_name}.jsonl"
write_jsonl(examples_path, [one_example])

variants_payload = json.loads(paths["variant_config"].read_text(encoding="utf-8"))
base_variant = next(
    variant for variant in variants_payload["variants"]
    if variant["variant_id"] == "full_inferred_contract"
)

variant = {
    **base_variant,
    "enabled": True,
    "variant_id": f"full_inferred_contract_{MODEL.replace('-', '_')}",
    "settings_overrides": {
        **base_variant.get("settings_overrides", {}),
        "data_understanding_model": f"deepseek:{MODEL}",
        "orchestrator_model": f"deepseek:{MODEL}",
        "evidence_model": f"deepseek:{MODEL}",
        "verifier_model": f"deepseek:{MODEL}",
        "writer_model": f"deepseek:{MODEL}",
        "auditor_model": f"deepseek:{MODEL}",
        "max_total_tokens": 300000,
        "max_agent_requests": 8,
    },
}

variants_path = project_dir / f"evaluation/config/variants_{experiment_name}.json"
variants_path.write_text(json.dumps({"variants": [variant]}, indent=2), encoding="utf-8")

generations_path = project_dir / f"evaluation/generations/{experiment_name}_generations.jsonl"
run_root = project_dir / f"evaluation/generations/{experiment_name}_runs"

print("Experiment:", experiment_name)
print("Workflow model:", MODEL)
print("Variant:", variant["variant_id"])
print("Running inferred-contract workflow...")

t0 = time.perf_counter()

generations = await generate_reports_for_notebook(
    project_dir,
    examples_path=examples_path,
    variants_path=variants_path,
    output_path=generations_path,
    run_root=run_root,
    resume=False,
)

print(f"\nGeneration finished in {time.perf_counter() - t0:.1f}s")

row = generations.iloc[0]
metadata = row.get("metadata") or {}

print("\nDone.")
print("Variant:", row["variant_id"])
print("Error:", row["error"])
print("Release status:", row.get("release_status"))
print("Writer mode:", row.get("writer_mode"))
print("Elapsed seconds:", row["elapsed_seconds"])
print("Pipeline result:", row.get("pipeline_result_path"))

print("\nInferred contract:")
print(json.dumps(metadata.get("inferred_task_contract"), indent=2))

print("\nResolved contract:")
print(json.dumps(metadata.get("resolved_task_contract"), indent=2))

print("\nAgreement:")
print(json.dumps(metadata.get("task_contract_agreement"), indent=2))

print("\nGenerated text:\n")
print(row["generated_text"])

display(
    generations[
        [
            "dataset_id",
            "example_id",
            "variant_id",
            "generated_text",
            "error",
            "release_status",
            "writer_mode",
            "elapsed_seconds",
        ]
    ]
)

# =========================
# Compare metrics with saved full_system/raw_generic_flash rows
# =========================

saved_generations_path = (
    project_dir
    / "evaluation/generations/five_dataset_five_each_raw_generic_flash_20260805_181001_combined_generations.jsonl"
)

saved_rows = []
if saved_generations_path.exists():
    for line in saved_generations_path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        saved_row = json.loads(line)
        if (
            saved_row.get("dataset_id") == dataset_id
            and str(saved_row.get("example_id")) == example_id
            and saved_row.get("variant_id") in {"full_system", "raw_generic_flash"}
        ):
            saved_rows.append(saved_row)

new_row = row.to_dict()

combined_generations_path = (
    project_dir
    / f"evaluation/generations/{experiment_name}_vs_saved_generations.jsonl"
)

with combined_generations_path.open("w", encoding="utf-8") as handle:
    for item in [*saved_rows, new_row]:
        handle.write(json.dumps(item, ensure_ascii=False) + "\n")

print("\nSaved comparison rows found:", len(saved_rows))
print("Combined generations:", combined_generations_path)

base_metrics_path = (
    project_dir
    / "evaluation/config/archive/metrics_five_dataset_five_each_raw_generic_flash_20260805_181001_reference.json"
)

metrics_payload = json.loads(base_metrics_path.read_text(encoding="utf-8"))
metrics_payload = {
    **metrics_payload,
    "experiment_id": experiment_name,
    "prepared_examples_path": str(examples_path),
    "generations_path": str(combined_generations_path),
    "baseline_variant": "raw_generic_flash",
}

metrics_path = project_dir / f"evaluation/config/metrics_{experiment_name}_reference.json"
metrics_path.write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")

scores_path = project_dir / f"evaluation/results/{experiment_name}_reference_metrics.jsonl"

print("\nScoring reference metrics...")
t0 = time.perf_counter()

scores = score_reference_metrics_for_notebook(
    project_dir,
    generations_path=combined_generations_path,
    metric_config_path=metrics_path,
    output_path=scores_path,
    include_ineligible=True,
)

print(f"Metrics finished in {time.perf_counter() - t0:.1f}s")
print("Metrics output:", scores_path)

metric_table = (
    scores[scores["status"].isin(["scored", "error", "skipped", "unavailable"])]
    .pivot_table(
        index=["dataset_id", "example_id", "metric_name"],
        columns="variant_id",
        values="score",
        aggfunc="first",
    )
    .reset_index()
)

display(metric_table)